# Auditoría final desde el ledger público

Este notebook recalcula las cifras finales de la memoria usando solo el ledger anonimizado publicado en `results/final_candidate_actions_anonymized.csv`.

No necesita el corpus privado ni los modelos entrenados. Por eso es el notebook público más directo para auditar el resultado final.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "results").exists():
    ROOT = ROOT.parent

ledger_path = ROOT / "results" / "final_candidate_actions_anonymized.csv"
summary_path = ROOT / "results" / "final_candidate_summary.csv"

ledger = pd.read_csv(ledger_path)
published = pd.read_csv(summary_path)
ledger.head()

## Resultado OOS limpio

El resultado confirmatorio es el especialista base sin el filtro de volatilidad posterior.

In [ ]:
base = ledger
base_table = pd.Series({
    "n_actions": len(base),
    "mean_net_0p25": base["net_cost_0p25"].mean(),
    "mean_net_0p5": base["net_cost_0p5"].mean(),
    "mean_net_1p0": base["net_cost_1p0"].mean(),
    "positive_days_0p5": (base.groupby("session_day")["net_cost_0p5"].mean() > 0).sum(),
    "n_days": base["session_day"].nunique(),
})
base_table

## Diagnóstico post-hoc de volatilidad

La separación por baja volatilidad es diagnóstica: se observó después de inspeccionar el bloque fresh y por tanto no debe leerse como una segunda validación OOS.

In [ ]:
regime_table = (
    ledger.groupby("volatility_regime")
    .agg(
        n=("action_id", "size"),
        net_0p25=("net_cost_0p25", "mean"),
        net_0p5=("net_cost_0p5", "mean"),
        net_1p0=("net_cost_1p0", "mean"),
        positive_days=("net_cost_0p5", lambda x: (ledger.loc[x.index].groupby("session_day")["net_cost_0p5"].mean() > 0).sum()),
    )
    .sort_index()
)
regime_table

## Incertidumbre agrupada por mercado

El bootstrap agrupado evita tratar como independientes varias señales solapadas del mismo mercado.

In [ ]:
BOOTSTRAP_SEED = 42
N_BOOTSTRAP = 5_000

low = ledger[ledger["volatility_regime"] == "low"].copy()
grouped = low.groupby("market_hash", sort=False)["net_cost_0p5"]
sums = grouped.sum().to_numpy(dtype=float)
counts = grouped.size().to_numpy(dtype=float)

rng = np.random.default_rng(BOOTSTRAP_SEED)
means = np.empty(N_BOOTSTRAP)
for i in range(N_BOOTSTRAP):
    sampled = rng.integers(0, len(sums), size=len(sums))
    means[i] = sums[sampled].sum() / counts[sampled].sum()

cluster_ci90 = np.quantile(means, [0.05, 0.95])
cluster_p_positive = (means > 0).mean()

pd.Series({
    "low_vol_actions": len(low),
    "unique_markets": low["market_hash"].nunique(),
    "max_actions_per_market": low.groupby("market_hash").size().max(),
    "market_cluster_ci90_low": cluster_ci90[0],
    "market_cluster_ci90_high": cluster_ci90[1],
    "market_cluster_p_positive": cluster_p_positive,
})

## Comparación con el resumen publicado

El script equivalente, apto para CI o consola, es:

```bash
python scripts/experiments/recompute_final_summary_from_public_ledger.py --check
```

In [ ]:
published.head(10)